In [1]:
!pip install ipdb -q
!pip install tqdm -q
!pip install sentencepiece -q
!pip install wandb -q

In [2]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
import os
import ipdb
from tqdm import tqdm
from datetime import datetime
import platform, shutil
import requests, zipfile, io

import torch
import torch.nn as nn
from torch.nn import functional as F

import sentencepiece as spm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.cuda.empty_cache()

In [4]:
files_url = "https://ideami.com/llm_train"

# Downloading proceeds if we detect that one of the key files to download is not present
if not os.path.exists(f"encoded_data.pt"):
    print("Downloading files using Python")
    response = requests.get(files_url)
    zipfile.ZipFile(io.BytesIO(response.content)).extractall(".")
else:
    print("you seem to have already downloaded the files. If you wish to re-download them, delete the encoded_data.pt file")

you seem to have already downloaded the files. If you wish to re-download them, delete the encoded_data.pt file


In [5]:
# PARAMETERS

## for collab batch_size = 32
batch_size = 8
context = 512
embed_size = 384
n_layers = 7
n_heads = 7
BIAS = True


# HYPERPARAMTERS
lr = 3e-4
dropout = 0.05
weight_decay = 0.01
grad_clip = 1.0

# TRAINING PARAMETERS
train_iters = 100000
eval_interval = 50
eval_iters = 10
compile = False
checkpoint_dir = 'models'
checkpoint_fn = 'latest.pt'
checkpoint_load_fn = 'latest.pt'
dtype = torch.bfloat16

# MODE
inference = False

# DEVICE
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device: You will be using: ",device)

device: You will be using:  cpu


In [6]:
# LOGGING
wandb_log = False  # Set to True only when you switch to T4 GPU for the real run!
# wandb_log = True
# wandb_project = "llm_test"
# wandb_run_name = "llm1" + datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

# if wandb_log:
#     import wandb
#     wandb.init(project=wandb_project, name=wandb_run_name)

In [7]:
with open('wiki.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(text[30000:30300])

terms.
For example, there are objects in two groups (as shown on the right). The objects are various shapes, where one group has 3 of them while the other has 2. When the two groups combine into one, the overall amount (sum) of the shapes become 5.

Vertical Addition

The animation above demonstrate


In [8]:
# TOKENIZER
sp = spm.SentencePieceProcessor(model_file='wiki_tokenizer.model')

vocab_size = sp.get_piece_size()
print("vocab_size: ",vocab_size)

# Create the encoding and decoding tokenizer functions
encode = lambda s: sp.Encode(s)
decode = lambda l: sp.Decode(l)


# Test that encoding and decoding are working well
print(decode(encode("Encoding Decoding functions ready")))

vocab_size:  4096
Encoding Decoding functions ready


In [9]:
# Tokenization of the dataset
if os.path.exists(f"encoded_data.pt"):
    # Load encoded data if saved it previously
    print("Loading saved encoded data")
    data = torch.load('encoded_data.pt')
else:
    # If still didn't encode and save the encoding, do it here
    print("Encoding data")
    data = torch.tensor(encode(text), dtype=torch.long)
    torch.save(data, 'encoded_data.pt')

Loading saved encoded data


In [10]:
data_size=len(data) # Get the size of the dataset

spl = int(0.9*data_size) # set the split at 90%-10%
train_data=data[:spl] # training data will be 90% of the dataset
val_data=data[spl:] # validation data will be 10% of the dataset
print(f'Total data: {data_size/1e6:.2f} Million | Training: {len(train_data)/1e6:.2f} Million | Validation: {len(val_data)/1e6:.2f} Million')

Total data: 59.21 Million | Training: 53.29 Million | Validation: 5.92 Million


In [11]:
def get_batch(split):
    # BS = Batch Size / SL = Sequence Length or context length
    data = train_data if split=="train" else val_data # Select the split
    inds = torch.randint(len(data)-context, (batch_size,)) # (BS)
    x = torch.stack([data[i: i+context] for i in inds]) # (BS,SL)
    y = torch.stack([data[i+1: i+context+1] for i in inds]) # (BS,SL)

    x,y = x.to(device), y.to(device)
    return x,y


# Uncomment to test your get_batch function
x,y=get_batch("train")
print(f"x.shape: {x.shape}")
print(f"y.shape: {y.shape}")
print(x[0][:10])
print(y[0][:10])

x.shape: torch.Size([8, 512])
y.shape: torch.Size([8, 512])
tensor([4079, 4062,  197,  163, 2856,  329, 4056, 4084, 4062,  197])
tensor([4062,  197,  163, 2856,  329, 4056, 4084, 4062,  197,  163])


In [13]:
from logging import log
class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.embeddings = nn.Embedding(vocab_size, embed_size)
    self.positions = nn.Embedding(context, embed_size)
    self.blocks = nn.Sequential(*[Block(n_heads) for _ in range(n_layers)])
    self.ln = nn.LayerNorm(embed_size)
    self.final_linear = nn.Linear(embed_size, vocab_size, bias=BIAS)
    self.apply(self._init_weights)


  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


  def forward(self, input, targets=None):
    loss = None
    BS,SL = input.shape
    emb = self.embeddings(input)
    pos = self.positions(torch.arange(SL, device=device))
    X = emb + pos
    X = self.blocks(X)
    X = self.ln(X)
    logits = self.final_linear(X)

    if targets is not None:
      BS, SL, VS = logits.shape
      logits = logits.view(BS*SL, VS)
      targets = targets.view(BS*SL)
      loss = F.cross_entropy(logits, targets)

      counts = logits.exp()
      prob = counts / counts.sum(-1, keepdim=True)





